# Benchmark SLM (prompts maison + HumanEval)

Notebook **commun aux 4 membres**. Vous ne changez que `MODEL_KEY` (cellule 3) et, une seule fois, `REPO_URL` / `REPO_NAME` (cellule 1).

Avant de lancer : `Exécution > Modifier le type d'exécution > T4 GPU`.

- **Partie A** : 15 prompts maison -> `results/benchmark_<modele>.csv`
- **Partie B** : 20 problèmes HumanEval -> `results/humaneval_<modele>.json`

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # un seul GPU, comme sur Colab
REPO_URL = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"

if not os.path.exists("../config.json"):
    !git clone {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

os.makedirs("../results", exist_ok=True)   # Git ne garde pas les dossiers vides

In [ ]:
!pip install -r ../requirements.txt -q
!nvidia-smi -L

## Gemma uniquement
Modèle "gated" : acceptez les conditions sur sa page Hugging Face, créez un token, puis décommentez. **Ne poussez jamais votre token sur GitHub.**

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
import json, time, sys, re
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

# === SEULE LIGNE A CHANGER : "qwen", "phi", "gemma" ou "smollm" ===
MODEL_KEY = "gemma"
# ====================================================================

with open("../config.json") as f:
    config = json.load(f)

MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)   # /no_think pour SmolLM3 seulement
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
print(MODEL_NAME, "|", model_slug, "|", GPU_NAME)

In [ ]:
torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
DTYPE = torch.bfloat16 if config["precision"] == "bf16" else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=DTYPE
)
print("Précision :", config["precision"])
model.eval()


def generate(user_prompt: str):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})

    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]

    # TTFT : temps pour produire 1 seul token
    torch.cuda.synchronize(); t0 = time.time()
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.time() - t0

    # Generation complete
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                         do_sample=GEN_PARAMS["do_sample"])
    torch.cuda.synchronize(); latency = time.time() - t0

    new_tokens = out[0][n_in:]
    return {
        "response": tokenizer.decode(new_tokens, skip_special_tokens=True),
        "gpu": GPU_NAME,
        "ttft_s": round(ttft, 3),
        "latency_s": round(latency, 3),
        "tokens_per_sec": round(len(new_tokens) / latency, 2) if latency > 0 else None,
        "input_tokens": n_in,
        "output_tokens": len(new_tokens),
        "gpu_memory_go": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


# Échauffement + contrôle : le 1er appel est lent, et on vérifie que le modèle répond vraiment
test = generate("Write a Python function add(a, b) that returns a + b.")
print(test["response"][:300])
assert "def" in test["response"], "Réponse vide ou incohérente : essayez bf16 dans config.json, puis redémarrez la session."
print("Modele pret.")

## Partie A : 15 prompts maison

In [ ]:
with open("../prompts.json") as f:
    prompts = json.load(f)["prompts"]

def norm(s):  # pour comparer sans tenir compte des espaces
    return re.sub(r"\s+", "", s)

results = []
for p in prompts:
    gen = generate(p["prompt"])
    response = gen.pop("response")

    status, error = None, None
    if p["task"] == "generation_code" and "test" in p:
        status, error = run_tests(extract_code(response), p["test"],
                                  timeout=HE_CONFIG["timeout_seconds"])
    elif p["task"] == "completion_code" and "expected" in p:
        status = "pass" if norm(p["expected"]) in norm(response) else "fail"
    # bug_detection, explication, refactoring, generation_tests : notes manuelles

    results.append({
        "id": p["id"], "task": p["task"], "prompt": p["prompt"], "response": response,
        **gen,
        "auto_status": status, "auto_error": error,
        # A remplir a la main (voir grille_notation.md et README)
        "bug_label": None, "note_qualite": None, "hallucination": None,
    })
    print(p["id"], status)

df_prompts = pd.DataFrame(results)
df_prompts.to_csv(f"../results/benchmark_{model_slug}.csv", index=False)
print(f"-> results/benchmark_{model_slug}.csv")
df_prompts[["id", "task", "ttft_s", "latency_s", "tokens_per_sec", "gpu_memory_go", "auto_status"]]

## Partie B : HumanEval (20 problèmes communs)

In [ ]:
with open("../humaneval_subset.json") as f:
    subset = json.load(f)

humaneval = load_dataset(HE_CONFIG["dataset"], split="test")
humaneval = humaneval.filter(lambda ex: ex["task_id"] in subset["task_ids"])
print(len(humaneval), "problemes")

HEADER = "from typing import *\nimport math\n\n"   # imports courants que le modele peut oublier

details = []
for ex in humaneval:
    gen = generate(HE_CONFIG["instruction"] + ex["prompt"])
    code_generated = extract_code(gen["response"])
    test_code = ex["test"] + f"\ncheck({ex['entry_point']})"
    status, error = run_tests(HEADER + ex["prompt"] + "\n" + code_generated, test_code,
                              timeout=HE_CONFIG["timeout_seconds"])
    details.append({
        "task_id": ex["task_id"], "status": status, "error": error,
        "ttft_s": gen["ttft_s"], "latency_s": gen["latency_s"],
        "tokens_per_sec": gen["tokens_per_sec"],
        "output_tokens": gen["output_tokens"], "gpu_memory_go": gen["gpu_memory_go"],
        "response": gen["response"],
    })
    print(ex["task_id"], status)

n_pass = sum(d["status"] == "pass" for d in details)
summary = {
    "model": MODEL_NAME,
    "gpu": GPU_NAME,
    "precision": config["precision"],
    "dataset": "HumanEval (sous-ensemble des 20 premiers problemes)",
    "n_problems": len(details),
    "n_pass": n_pass,
    "pass@1": round(100 * n_pass / len(details), 1),
    "details": details,
}
with open(f"../results/humaneval_{model_slug}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"pass@1 = {summary['pass@1']}% ({n_pass}/{len(details)}) -> results/humaneval_{model_slug}.json")

In [ ]:
# Fichier d'environnement PROPRE A CHAQUE MEMBRE (pas d'ecrasement entre vous)
from importlib.metadata import version
import platform
lines = [f"# Environnement - {MODEL_NAME}", "",
         f"- GPU : {GPU_NAME}",
         f"- Python : {platform.python_version()}",
         f"- torch : {torch.__version__}",
         f"- CUDA : {torch.version.cuda}"]
for lib in ["transformers", "accelerate", "huggingface_hub", "datasets", "pandas"]:
    lines.append(f"- {lib} : {version(lib)}")
open(f"../environnement_{model_slug}.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))